# PyBaMM solvers

Every PyBaMM model is solved by `pybamm.IDAKLUSolver`, a wrapper around the
SUNDIALS IDAS solver with the KLU sparse linear solver, written by the PyBaMM
team. It is the default for every model, so you rarely need to create one
yourself: you create one only to change its settings.

The older `CasadiSolver` and `ScipySolver` are deprecated, and new features
only go into IDAKLU. This notebook covers the IDAKLU settings that matter in
practice: how it steps in time, tolerances, output variables, parallel solves,
and sensitivities.

In [1]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

import pybamm

In [2]:
type(pybamm.lithium_ion.DFN().default_solver)

pybamm.solvers.idaklu_solver.IDAKLUSolver

## 1. Time stepping: `t_eval` and `t_interp`

The solver chooses its own internal time steps to meet its error tolerances,
starting small and growing as it becomes confident in the solution.

In [3]:
sim = pybamm.Simulation(pybamm.lithium_ion.SPM())
sol = sim.solve([0, 10])
print("internal time steps:", sol.t)

internal time steps: [ 0.          0.01        0.02        0.04        0.08        0.16
  0.32000001  0.64000001  1.28000002  2.56000004  3.84000006  5.12000008
  7.68000013 10.        ]


Every point in `t_eval` forces the solver to **stop and restart**, which is
expensive. Only pass more than a start and end time if there is a
discontinuity the solver cannot know about.

If you just want the output at particular times, pass them as `t_interp`
instead: the solver interpolates as it goes, without stopping.

In [4]:
sol = sim.solve(t_eval=np.array([0, 5, 10]))
print("t_eval=[0, 5, 10]:", sol.t)
sol = sim.solve([0, 10], t_interp=[2, 4, 6, 8])
print("t_interp=[2, 4, 6, 8]:", sol.t)

t_eval=[0, 5, 10]: [ 0.          0.01        0.02        0.04        0.08        0.16
  0.32000001  0.64000001  1.28000002  2.56000004  3.84000006  5.
  5.005       5.01        5.02        5.04        5.08        5.16000001
  5.32000001  5.64000002  6.28000004  7.56000008 10.        ]
t_interp=[2, 4, 6, 8]: [ 0.  2.  4.  6.  8. 10.]


`t_interp` also controls what is **stored**: without it the solution keeps
every internal step, with it only the times you asked for. In an experiment,
the `period` of a step plays the same role.

In [5]:
parameter_values = pybamm.ParameterValues("Chen2020")
model = pybamm.lithium_ion.DFN()
sim = pybamm.Simulation(model, parameter_values=parameter_values)
sol = sim.solve([0, 3600])
print("stored points, internal steps:", len(sol.t))
sol = sim.solve([0, 3600], t_interp=np.linspace(0, 3600, 13))
print("stored points, t_interp:      ", len(sol.t))

experiment = pybamm.Experiment(["Discharge at 1C for 1 hour (5 minute period)"])
sol = pybamm.Simulation(
    model, parameter_values=parameter_values, experiment=experiment
).solve()
print("stored points, 5 min period:  ", len(sol.t))

stored points, internal steps: 170
stored points, t_interp:       13


stored points, 5 min period:   13


## 2. Tolerances

The solver accepts a step when its error estimate $e$ satisfies

$$
\sqrt{\frac{1}{n} \sum_{i=1}^n \left( \frac{e_i}{\epsilon_r |y_i| + \epsilon_a} \right)^2 } \leq 1,
$$

with relative tolerance $\epsilon_r$ (`rtol`, default $10^{-4}$) and absolute
tolerance $\epsilon_a$ (`atol`, default $10^{-6}$). Tighter tolerances are more
accurate and slower. A convergence study against a very tight solution shows
where the trade-off lies:

In [6]:
model = pybamm.lithium_ion.DFN()
t = np.linspace(0, 3600, 100)
reference = pybamm.Simulation(
    model, solver=pybamm.IDAKLUSolver(rtol=1e-10, atol=1e-10)
).solve([0, 3600])["Voltage [V]"](t)

tolerances = [1e-2, 1e-3, 1e-4, 1e-6, 1e-8]
errors, times = [], []
for tol in tolerances:
    sim = pybamm.Simulation(model, solver=pybamm.IDAKLUSolver(rtol=tol, atol=tol))
    sim.solve([0, 3600])  # build and warm up
    start = time.perf_counter()
    voltage = sim.solve([0, 3600])["Voltage [V]"](t)
    times.append(time.perf_counter() - start)
    errors.append(np.max(np.abs(voltage - reference)))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].loglog(tolerances, np.array(errors) * 1000, "o-")
ax[0].set_ylabel("Max voltage error [mV]")
ax[1].semilogx(tolerances, np.array(times) * 1000, "o-")
ax[1].set_ylabel("Solve time [ms]")
for a in ax:
    a.set_xlabel("rtol = atol")
plt.tight_layout()

## 3. Only compute what you need: `output_variables`

By default the solver stores the full state vector so that any variable can be
computed afterwards. If you know which outputs you want, list them: the solver
computes them on the fly and skips storing the state, which saves memory and
time for big models and long experiments.

In [7]:
for output_variables in [None, ["Voltage [V]", "Current [A]"]]:
    solver = pybamm.IDAKLUSolver(output_variables=output_variables)
    sim = pybamm.Simulation(model, solver=solver)
    sim.solve([0, 3600])
    start = time.perf_counter()
    sol = sim.solve([0, 3600])
    sol["Voltage [V]"](t)
    print(f"output_variables={output_variables}: {time.perf_counter() - start:.4f} s")

output_variables=None: 0.0165 s
output_variables=['Voltage [V]', 'Current [A]']: 0.0130 s


## 4. Many solves at once

Combined with input parameters (previous notebook), passing a **list** of
inputs solves them all in one call, in parallel across `num_threads` threads.

On Windows, more than one thread can crash the kernel
([PyBaMM#5722](https://github.com/pybamm-team/PyBaMM/issues/5722)), so
this cell uses one thread there.

In [8]:
parameter_values = pybamm.ParameterValues("Chen2020")
parameter_values["Current function [A]"] = "[input]"
inputs = [{"Current function [A]": current} for current in np.linspace(1, 10, 16)]

# More than one thread can crash the kernel on Windows (PyBaMM#5722)
thread_counts = [1] if sys.platform == "win32" else [1, 4]
for num_threads in thread_counts:
    solver = pybamm.IDAKLUSolver(options={"num_threads": num_threads})
    sim = pybamm.Simulation(model, parameter_values=parameter_values, solver=solver)
    sim.solve([0, 3600], inputs=inputs[0])
    start = time.perf_counter()
    solutions = sim.solve([0, 3600], inputs=inputs)
    print(
        f"{num_threads} thread(s): {time.perf_counter() - start:.3f} s for {len(solutions)} solves"
    )

1 thread(s): 0.158 s for 16 solves
4 thread(s): 0.045 s for 16 solves


In [9]:
fig, ax = plt.subplots()
for sol, inp in zip(solutions, inputs):
    ax.plot(
        sol.t / 60,
        sol["Voltage [V]"](sol.t),
        label=f"{inp['Current function [A]']:.1f} A",
    )
ax.set_xlabel("Time [min]")
ax.set_ylabel("Voltage [V]")

Text(0, 0.5, 'Voltage [V]')

## 5. Sensitivities

IDAKLU can compute the derivative of every output with respect to input
parameters alongside the solution, which is what gradient-based parameter
fitting needs.

In [10]:
parameter_values = pybamm.ParameterValues("Chen2020")
parameter_values["Negative particle diffusivity [m2.s-1]"] = "[input]"
sim = pybamm.Simulation(model, parameter_values=parameter_values)
sol = sim.solve(
    [0, 3600],
    inputs={"Negative particle diffusivity [m2.s-1]": 3.3e-14},
    calculate_sensitivities=True,
)
sensitivity = sol["Voltage [V]"].sensitivities["Negative particle diffusivity [m2.s-1]"]
plt.plot(sol.t / 60, sensitivity * 3.3e-14)
plt.xlabel("Time [min]")
plt.ylabel("D dV/dD [V]")

Text(0, 0.5, 'D dV/dD [V]')

## Exercises

1. Run a 100-cycle experiment from Day 1 with `output_variables` set. How much
   faster is it?
2. Use a list of inputs to sweep the positive particle radius, and find the
   largest radius that still delivers 90% of the 1C capacity.
3. Which parameter is the voltage most sensitive to at the end of a 1C
   discharge? Compare normalised sensitivities for a few candidates.

See the [IDAKLUSolver docs](https://docs.pybamm.org/en/stable/source/api/solvers/idaklu_solver.html)
for every option.